# Karllo AI — training notebook

Use this notebook when the website host cannot train. It runs **real** PyTorch training.

Free Colab GPUs are quota-limited and often unavailable. This notebook works on CPU for the tiny/small configs.

After training, download the checkpoint folder and copy it into `checkpoints/runs/` on the Karllo server, then **verify** and **manually activate** it in the UI. This never overwrites the original init checkpoint.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'backend' / 'karllo').exists():
    print('Run from the Karllo repository root, or upload the repo to Colab.')
sys.path.insert(0, str(ROOT / 'backend'))
print('ROOT', ROOT)

In [ ]:
from karllo.hardware import hardware_report, estimate_run
from karllo.model.config import KarlloConfig
hw = hardware_report()
print(hw)
cfg = KarlloConfig.tiny()
print('tiny params (est)', cfg.estimated_parameters())
print(estimate_run(cfg.estimated_parameters(), 20_000, hw['device']))

In [ ]:
from karllo.training.engine import TrainConfig, run_training
from pathlib import Path
corpus = ROOT / 'data' / 'seed' / 'pilot_corpus.txt'
cfg = TrainConfig(
    mode='pretrain',
    dataset_file=str(corpus),
    run_dir=str(ROOT / 'data' / 'runs' / 'colab-pilot'),
    checkpoint_out=str(ROOT / 'checkpoints' / 'runs' / 'colab-pilot'),
    model_size='tiny',
    vocab_size=512,
    max_steps=40,
    batch_size=2,
    grad_accum=2,
    lr=3e-3,
    device=hw['device'],
    train_tokenizer=True,
)
status = run_training(cfg)
print(status['state'], 'loss', status.get('train_loss'), 'changed', status.get('trained_weights_changed'))
print('checkpoint', status.get('latest_checkpoint'))

## Integrate the checkpoint

1. Zip and download `checkpoints/runs/colab-pilot/latest`.
2. Place it under `checkpoints/runs/<id>/latest` on the server.
3. In Karllo: Checkpoints → Verify → Evaluate → Activate (manual).
4. The original random init remains untouched.